# 26. Segmentation Test (Pipeline 06)

- Goal: inspect phase segmentation readiness and confirm that annotation rep labels survive the stage handoff.
- Docs: `docs_eng/pipeline/06_segmentation.md` / `docs/pipeline/06_segmentation.md`
- Prerequisite: 20-25 stage checks should already pass.
- Input: normalized preprocessed pose dataframe from the same previous-stage setup used by 24-25; canonical analysis-space coordinates from 25 are not required for this boundary/phase-label check.
- Output: frame-level `phase` labels and segmentation reports/provenance; existing annotation `rep_id` labels are preserved.
- Checks: previous-stage setup, compact rep handoff smoke check, direct phase segmentation, label coverage, non-rep handling, report structure, inflection summary, and pipeline integration.
- Policy: recording-specific phase-split guides are QC references only; they are not ground truth labels or scoring inputs.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd

from movement.config import LANDMARKS
from movement.normalization import check_normalization_result
from movement.pipeline import (
    NormalizationConfig,
    PreprocessingConfig,
    run_pipeline,
)
from movement.segmentation import (
    PhaseSegmentationReport,
    RepSegmentationReport,
    segment_phases,
    segment_reps,
)
from movement.stage_context import (
    build_stage_check_pipeline_config,
    find_project_root,
    prepare_previous_stage_inputs,
)

print('imports OK')


## Data Setup

Runs the previous context in pipeline order before segmentation:

```text
Pose CSV -> ① Validation -> ② Annotation -> ③ Exercise Definition -> ④ Preprocessing -> ⑤ Normalization
```

In [ ]:
PROJECT_ROOT = find_project_root()

pose_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_output_pose.csv'
annotation_csv = 'data/pose/mediapipe/no_consent/20260517/p01_squat_set1_annotation.csv'
TARGET_EXERCISE_ID = 'squat'

pre_config = PreprocessingConfig(enabled=True)
norm_config = NormalizationConfig(
    enabled=True,
    keep_reference_columns=True,
    model_depth_scale=1.0,
)
stage_inputs = prepare_previous_stage_inputs(
    prepare_until='normalization',
    pose_csv=pose_csv,
    annotation_csv=annotation_csv,
    exercise_id=TARGET_EXERCISE_ID,
    landmarks=LANDMARKS,
    preprocessing_config=pre_config,
    normalization_config=norm_config,
)

df_raw = stage_inputs.raw_df
ann_df = stage_inputs.annotation_df
val_report = stage_inputs.validation_report
ann_report = stage_inputs.annotation_report
exercise_def = stage_inputs.exercise_definition
pre_df = stage_inputs.preprocessed_df
pre_report = stage_inputs.preprocessing_report
norm_df = stage_inputs.normalized_df
norm_report = stage_inputs.normalization_report
annotation_path = stage_inputs.annotation_csv
TARGET_DEFINITIONS_DIR = stage_inputs.definitions_dir

check_report = check_normalization_result(norm_df)
assert check_report['passed'] is True
assert exercise_def.exercise_id == TARGET_EXERCISE_ID
assert exercise_def.rep_segmentation is not None, 'exercise definition must have rep_segmentation block'
assert exercise_def.phase_segmentation is not None, 'exercise definition must have phase_segmentation block'

setup_summary = pd.DataFrame([
    {'item': 'frames_loaded', 'value': len(df_raw)},
    {'item': 'validation_passed', 'value': val_report['passed']},
    {'item': 'structural_validation_passed', 'value': val_report.get('structural_passed')},
    {'item': 'analysis_frames', 'value': f"{ann_report['num_analysis_frames']} / {ann_report['num_total_frames']}"},
    {'item': 'exercise_id', 'value': exercise_def.exercise_id},
    {'item': 'movement_template_id', 'value': exercise_def.classification['movement_template_id']},
    {'item': 'preprocessing_invalid_frames', 'value': pre_report['num_invalid_frames']},
    {'item': 'normalization_scale_value', 'value': round(float(norm_report['scale_value']), 6)},
    {'item': 'normalized_shape', 'value': norm_df.shape},
    {'item': 'rep_reference', 'value': f"{exercise_def.rep_segmentation.reference_landmark}:{exercise_def.rep_segmentation.reference_coordinate_family}:{exercise_def.rep_segmentation.reference_axis}"},
    {'item': 'phase_reference', 'value': f"{exercise_def.phase_segmentation.reference_landmark}:{exercise_def.phase_segmentation.reference_coordinate_family}:{exercise_def.phase_segmentation.reference_axis}"},
    {'item': 'definitions_dir', 'value': str(TARGET_DEFINITIONS_DIR)},
])
display(setup_summary)

if val_report.get('warnings'):
    display(pd.DataFrame(val_report['warnings']))
if not val_report['passed']:
    print('NOTE: structural validation failed; inspect before segmentation.')


## Recording-Specific Phase Guideline

`p01_squat_set1_phase_split.csv`, when present, is a visual/QC guide for this recording only. It is not ground truth phase annotation and is not used as pipeline scoring input.

In [ ]:
phase_guideline_path = annotation_path.with_name('p01_squat_set1_phase_split.csv')
phase_guideline_df = pd.DataFrame()

if phase_guideline_path.exists():
    phase_guideline_df = pd.read_csv(phase_guideline_path)
    print('phase guideline:', phase_guideline_path.relative_to(PROJECT_ROOT))
    print('policy: guide only; not confirmed annotation; not scoring input')
    print('rows:', len(phase_guideline_df))
    print('reps:', sorted(phase_guideline_df['rep_id'].dropna().astype(int).unique().tolist()))

    rep_ranges = (
        ann_df.loc[ann_df['segment_type'].eq('rep'), ['set_id', 'rep_id', 'start_frame', 'end_frame']]
        .rename(columns={'start_frame': 'annotation_start', 'end_frame': 'annotation_end'})
        .copy()
    )
    guide_ranges = (
        phase_guideline_df.groupby(['set_id', 'rep_id'], dropna=False)
        .agg(
            guideline_start=('start_frame', 'min'),
            guideline_end=('end_frame', 'max'),
            guideline_phases=('phase', lambda s: ' / '.join(s.astype(str))),
        )
        .reset_index()
    )
    coverage = rep_ranges.merge(guide_ranges, on=['set_id', 'rep_id'], how='left')
    coverage['covers_annotation_rep'] = (
        coverage['annotation_start'].eq(coverage['guideline_start'])
        & coverage['annotation_end'].eq(coverage['guideline_end'])
    )
    display(coverage)
    display(phase_guideline_df.head(12))
else:
    print('No phase guideline CSV found beside the annotation file.')
    print('Pipeline phase segmentation will be inspected without recording-specific phase guidance.')

## Direct Handoff + Phase Segmentation Test

Call `segment_reps()` only as a compact handoff check. Existing annotation `rep_id` labels should be preserved, then `segment_phases()` fills labels inside those reps.

In [ ]:
df_rep_seg, rep_report = segment_reps(norm_df, exercise_def, fps_default=30.0)
df_seg, reports = segment_phases(df_rep_seg, exercise_def, fps_default=30.0)

rep_mask = df_seg['segment_type'] == 'rep'
phase_report_df = pd.DataFrame([r.as_dict() for r in reports])
rejected_count = int(phase_report_df['rejected_reason'].notna().sum()) if not phase_report_df.empty else 0

direct_summary = pd.DataFrame([
    {'item': 'rep_handoff_status', 'value': rep_report.status},
    {'item': 'rep_handoff_source', 'value': rep_report.source},
    {'item': 'rep_assignments_preserved', 'value': len(rep_report.rep_assignments)},
    {'item': 'phase_reports', 'value': len(reports)},
    {'item': 'phase_rejected_reps', 'value': rejected_count},
    {'item': 'phase_labeled_rep_frames', 'value': f"{int(df_seg.loc[rep_mask, 'phase'].notna().sum())} / {int(rep_mask.sum())}"},
])
display(direct_summary)

## Check 1: Rep Handoff Smoke Check

This is not a second annotation audit. It only verifies that segmentation preserves existing annotation rep labels instead of replacing them.

In [ ]:
assert isinstance(rep_report, RepSegmentationReport)
assert rep_report.status == 'manual_override'
assert rep_report.source == 'annotation'

rep_mask_handoff = df_rep_seg['segment_type'] == 'rep'
rep_ids = sorted(df_rep_seg.loc[rep_mask_handoff, 'rep_id'].dropna().astype(int).unique().tolist())
assert len(rep_ids) == 10
assert set(df_rep_seg.loc[rep_mask_handoff, 'rep_segmentation_status'].unique()) == {'manual_override'}
assert set(df_rep_seg.loc[rep_mask_handoff, 'rep_segmentation_source'].unique()) == {'annotation'}
print(f'PASS: rep handoff preserved {len(rep_ids)} annotation reps ({rep_ids[0]}-{rep_ids[-1]})')

## Check 2: Phase Labels Are Exercise-Defined

Rep frames should receive labels from the exercise definition, plus `Turnaround_Hold` when the hold option is enabled.

In [ ]:
assert 'phase' in df_seg.columns, 'phase column missing'

rep_mask = df_seg['segment_type'] == 'rep'
nrep = int(rep_mask.sum())
n_labeled = int(df_seg.loc[rep_mask, 'phase'].notna().sum())
print(f'rep frames: {nrep}  labeled: {n_labeled}')
if n_labeled < nrep:
    print('NOTE: phase labels are incomplete; inspect PhaseSegmentationReport rejection reasons below.')

valid_labels = set(exercise_def.phase_segmentation.phase_sequence)
if exercise_def.phase_segmentation.turnaround_hold.enabled:
    valid_labels.add('Turnaround_Hold')
actual = set(df_seg.loc[rep_mask, 'phase'].dropna().unique())
print(f'valid labels: {valid_labels}')
print(f'labels found: {actual}')
assert actual.issubset(valid_labels), f'unexpected labels: {actual - valid_labels}'
assert set(exercise_def.phase_segmentation.phase_sequence).issubset(actual)
assert n_labeled == nrep

counts = df_seg.loc[rep_mask, 'phase'].value_counts()
display(counts.rename('frames').to_frame())
print('PASS: all rep frames have exercise-defined phase labels')

## Check 3: Non-rep Frames Remain NA

In [ ]:
non_rep_mask = df_seg['segment_type'] != 'rep'
n_non_rep = int(non_rep_mask.sum())
n_non_rep_na = int(df_seg.loc[non_rep_mask, 'phase'].isna().sum())
print(f'non-rep frames: {n_non_rep}  still NA: {n_non_rep_na}')
assert n_non_rep_na == n_non_rep, 'non-rep frames must have NA phase'
print('PASS: non-rep frames remain NA')

## Check 4: Segmentation Report Structure

In [ ]:
for r in reports:
    assert isinstance(r, PhaseSegmentationReport)
    assert isinstance(r.rep_id, int)
    assert isinstance(r.inflection_frames, list)
    assert isinstance(r.phase_assignments, dict)
    d = r.as_dict()
    assert 'rep_id' in d
    assert 'inflection_frames' in d
    assert 'phase_assignments' in d
    assert 'smoothing_method' in d

report_df = phase_report_df.copy()
compact_report = report_df[['rep_id', 'rejected_reason', 'inflection_frames', 'multi_inflection_collapsed']]
display(compact_report)
assert report_df['rejected_reason'].isna().all()
print(f'PASS: PhaseSegmentationReport structure valid for all {len(reports)} reps')

## Check 5: Inflection Count And Rejection Summary

With `multi_inflection_policy=global_extremum`, each usable two-phase rep should collapse to one selected inflection frame. Collapsed multi-inflection reps remain provenance, not a failure.

In [ ]:
inflection_counts = [len(r.inflection_frames) for r in reports]
summary = pd.Series(inflection_counts, name='num_inflections').value_counts().sort_index()
collapsed_reps = [r.rep_id for r in reports if r.multi_inflection_collapsed]
display(summary.to_frame())
print(f'multi-inflection collapsed reps: {collapsed_reps or "none"}')
assert set(summary.index) == {1}
print('PASS: each accepted rep has one selected inflection')

## Check 6: Pipeline Integration

Run the same stage through `run_pipeline()` and compare report presence and phase-label coverage.

In [ ]:
cfg = build_stage_check_pipeline_config(
    exercise_id=TARGET_EXERCISE_ID,
    definitions_dir=TARGET_DEFINITIONS_DIR,
    annotation_csv=annotation_path,
    preprocessing_config=pre_config,
    normalization_config=norm_config,
    enable_rep_segmentation=True,
    enable_phase_segmentation=True,
)

pipe_df, pipe_report = run_pipeline(
    df_raw,
    config=cfg,
    landmarks=LANDMARKS,
    ann_df=ann_df,
)

assert 'rep_segmentation' in pipe_report, 'rep_segmentation key missing in report'
assert 'phase_segmentation' in pipe_report, 'phase_segmentation key missing in report'
ps_report = pipe_report['phase_segmentation']
assert len(ps_report) == len(reports), 'pipeline/direct phase report count mismatch'

rep_mask_pipe = pipe_df['segment_type'] == 'rep'
n_labeled_pipe = int(pipe_df.loc[rep_mask_pipe, 'phase'].notna().sum())
print(f"pipeline ⑥ rep_segmentation status: {pipe_report['rep_segmentation']['status']}")
print(f'pipeline ⑥ phase_segmentation report: {len(ps_report)} reps')
print(f'phase column labeled for {n_labeled_pipe} / {int(rep_mask_pipe.sum())} rep frames')
print(f'steps executed: {list(pipe_report.keys())}')
assert pipe_report['exercise_definition']['exercise_id'] == TARGET_EXERCISE_ID
assert n_labeled_pipe == int(rep_mask_pipe.sum())
print('PASS: pipeline integration matches direct segmentation coverage')


## Check Summary

This notebook is a compact execution/QC checkpoint for ⑥ Segmentation. Use the pipeline document linked in the header for definitions, interpretation policy, and scope.